# 04 - Agentes com ferramentas (Tool Calling) com LangGraph

Caso prático: um **agente de compras**. Ele recebe uma pergunta sobre preços, consulta uma API de preços, faz as contas e responde com base nos resultados.

Fluxo:

```
START → agente → (o LLM pediu ferramenta?) → ferramentas → agente → ... → END
                       └── não: responde e termina
```

- **Ferramenta (`@tool`)**: função Python com nome, descrição e tipos. O LLM vê essa descrição e decide quando usá-la.
- **`bind_tools`**: envia as ferramentas ao LLM junto com a conversa.
- **`ToolNode`**: executa a ferramenta que o LLM pediu e devolve o resultado como `ToolMessage`.
- **`tools_condition`**: aresta condicional pronta. Se a última mensagem tem `tool_calls`, vai para as ferramentas; senão, termina.
- O laço `ferramentas → agente` deixa o LLM ver o resultado e decidir o próximo passo: outra ferramenta ou a resposta final.

Conceitos do notebook anterior que continuam aqui: `MessagesState` e arestas condicionais.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

In [1]:
import ast
import json
import operator
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 2. Ferramenta 1: consulta de preços

A ferramenta chama uma **API de preços**. Para o notebook rodar sem depender de internet, a API é simulada com um dicionário local. Na prática, a função faria algo como `requests.get(...)` e devolveria o JSON da API.

A docstring vira a descrição que o LLM lê. Os tipos dos parâmetros viram o esquema de entrada.

In [2]:
PRECOS_API = {
    "café": 6.50,
    "pão": 1.20,
    "queijo": 22.00,
    "leite": 5.40,
}


@tool
def consultar_preco(produto: str) -> str:
    """Consulta na API de preços o preço unitário de um produto, em reais. Retorna JSON."""
    preco = PRECOS_API.get(produto.lower().strip())
    if preco is None:
        return json.dumps({"produto": produto, "erro": "produto não encontrado"}, ensure_ascii=False)
    return json.dumps({"produto": produto, "preco": preco, "moeda": "BRL"}, ensure_ascii=False)


print(consultar_preco.name)
print(consultar_preco.args)

consultar_preco
{'produto': {'title': 'Produto', 'type': 'string'}}


## 3. Ferramenta 2: calculadora

Poderíamos usar `eval`, mas ele executaria qualquer código que o LLM gerasse. Em vez disso, lemos a expressão com `ast` e aceitamos só números e as quatro operações.

In [3]:
OPERACOES = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def _avaliar(no: ast.AST) -> float:
    if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
        return no.value
    if isinstance(no, ast.BinOp) and type(no.op) in OPERACOES:
        return OPERACOES[type(no.op)](_avaliar(no.left), _avaliar(no.right))
    if isinstance(no, ast.UnaryOp) and isinstance(no.op, ast.USub):
        return -_avaliar(no.operand)
    raise ValueError("expressão não permitida")


@tool
def calcular(expressao: str) -> str:
    """Calcula uma expressão aritmética com números, + - * / e parênteses. Exemplo: '3 * 6.5 + 2 * 1.2'."""
    return str(round(_avaliar(ast.parse(expressao, mode="eval").body), 2))


print(calcular.invoke({"expressao": "3 * 6.5 + 2 * 1.2"}))

21.9


## 4. Montando o agente

- `llm_com_tools` é o mesmo LLM, mas com as ferramentas anexadas.
- O nó `agente` põe uma instrução de sistema na frente das mensagens. A instrução não é salva no estado.
- O nó `ferramentas` é um `ToolNode`, que executa as chamadas pedidas pelo LLM.

In [4]:
FERRAMENTAS = [consultar_preco, calcular]
llm_com_tools = llm.bind_tools(FERRAMENTAS)

INSTRUCAO = (
    "Você é um assistente de compras. Use as ferramentas para consultar preços e fazer contas. "
    "Não invente preços. Se um produto não for encontrado, diga isso. Responda em português."
)


def agente(estado: MessagesState) -> dict:
    resposta = llm_com_tools.invoke([SystemMessage(content=INSTRUCAO)] + estado["messages"])
    return {"messages": [resposta]}


grafo = StateGraph(MessagesState)
grafo.add_node("agente", agente)
grafo.add_node("ferramentas", ToolNode(FERRAMENTAS))

grafo.add_edge(START, "agente")
grafo.add_conditional_edges("agente", tools_condition, {"tools": "ferramentas", END: END})
grafo.add_edge("ferramentas", "agente")

app = grafo.compile()

## 5. Executando

A função `perguntar` roda o agente e mostra o caminho das mensagens:

- `→` é uma chamada de ferramenta pedida pelo LLM.
- `←` é o resultado devolvido pela ferramenta.
- `Resposta` é a mensagem final, sem chamadas.

In [5]:
def perguntar(texto: str) -> None:
    estado = app.invoke({"messages": [HumanMessage(content=texto)]})

    linhas = [f"**Pergunta:** {texto}", ""]
    for mensagem in estado["messages"][1:]:
        if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
            for chamada in mensagem.tool_calls:
                linhas.append(f"→ `{chamada['name']}({chamada['args']})`")
        elif isinstance(mensagem, ToolMessage):
            linhas.append(f"← `{mensagem.content}`")
        elif isinstance(mensagem, AIMessage):
            linhas.append(f"")
            linhas.append(f"**Resposta:** {mensagem.content}")
    display(Markdown("\n".join(linhas)))


perguntar("Quanto custam 3 cafés e 2 pães? Aplique 10% de desconto no total.")

**Pergunta:** Quanto custam 3 cafés e 2 pães? Aplique 10% de desconto no total.

→ `consultar_preco({'produto': 'café'})`
→ `consultar_preco({'produto': 'pão'})`
← `{"produto": "café", "preco": 6.5, "moeda": "BRL"}`
← `{"produto": "pão", "preco": 1.2, "moeda": "BRL"}`
→ `calcular({'expressao': '3 * 6.5 + 2 * 1.2'})`
← `21.9`
→ `calcular({'expressao': '21.9 * 0.9'})`
← `19.71`

**Resposta:** Consultei os preços. Valores unitários:
- Café: R$ 6,50
- Pão: R$ 1,20

Cálculo:
- 3 cafés = 3 × R$ 6,50 = R$ 19,50
- 2 pães = 2 × R$ 1,20 = R$ 2,40
- Subtotal = R$ 19,50 + R$ 2,40 = R$ 21,90
- Desconto de 10% = R$ 2,19
- Total com desconto = R$ 21,90 − R$ 2,19 = R$ 19,71

Total a pagar: R$ 19,71.

In [6]:
perguntar("Quanto custam queijo e leite juntos?")

**Pergunta:** Quanto custam queijo e leite juntos?

→ `consultar_preco({'produto': 'queijo'})`
→ `consultar_preco({'produto': 'leite'})`
← `{"produto": "queijo", "preco": 22.0, "moeda": "BRL"}`
← `{"produto": "leite", "preco": 5.4, "moeda": "BRL"}`
→ `calcular({'expressao': '22.0 + 5.4'})`
← `27.4`

**Resposta:** O total é R$ 27,40 (queijo R$ 22,00 + leite R$ 5,40).

In [7]:
perguntar("Quanto custa uma banana?")

**Pergunta:** Quanto custa uma banana?

→ `consultar_preco({'produto': 'banana'})`
← `{"produto": "banana", "erro": "produto não encontrado"}`

**Resposta:** Não encontrei preço para “banana” na base de dados. Não invento valores — quer que eu tente buscar uma variedade/descrição mais específica (ex.: banana nanica, prata, por kg ou por unidade) ou em alguma loja/cidade em particular?

## 6. Conferindo o estado final

Como o estado é um `MessagesState`, a conversa inteira fica em `messages`, com cada chamada e cada resultado. Isso permite conferir o raciocínio do agente depois.

In [8]:
estado = app.invoke({"messages": [HumanMessage(content="Quanto custa 2 leites?")]})

for mensagem in estado["messages"]:
    print(f"{mensagem.type:<6} | {mensagem.content!s:.80}")
    if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
        print(f"       | tool_calls: {[c['name'] for c in mensagem.tool_calls]}")

human  | Quanto custa 2 leites?
ai     | 
       | tool_calls: ['consultar_preco']
tool   | {"produto": "leite", "preco": 5.4, "moeda": "BRL"}
ai     | 
       | tool_calls: ['calcular']
tool   | 10.8
ai     | 2 leites custam R$ 10,80.


## Observações

- A **descrição** da ferramenta é o que o LLM usa para decidir quando chamá-la. Descrições vagas levam a chamadas erradas.
- O LLM não executa nada. Ele só pede a chamada, e o `ToolNode` executa.
- O laço `ferramentas → agente` permite encadear chamadas: primeiro o preço, depois a conta.
- Nunca use `eval` em ferramenta que recebe texto do LLM. A versão com `ast` aceita só aritmética.
- Para trocar a API simulada por uma real, mude só o corpo de `consultar_preco`. O resto do grafo fica igual.